# TAG_LENS - Colab GPU 재학습

장르 5-class(`자연/도시/음식/사물/인물`) CNN과 주간/야간 이진 CNN을 GPU로 재학습합니다.
로컬 `scripts/train_genre_model.py` / `scripts/train_genre_model_phase2.py` / `scripts/train_daynight_model.py`와 동일한 구조·하이퍼파라미터를 그대로 씁니다.

**실행 전 준비**
1. 상단 메뉴 `런타임 > 런타임 유형 변경`에서 하드웨어 가속기를 **GPU(T4)** 로 설정
2. `dataset.zip`, `dataset_daynight.zip`을 Google Drive의 `MyDrive/tag_lens_training/` 폴더에 업로드
3. 아래 셀을 위에서부터 순서대로 실행

다 끝나면 `MyDrive/tag_lens_training/genre_model.h5`, `daynight_model.h5`가 생기니 그걸 로컬 `tag_lens/models/`로 다운로드해서 덮어쓰면 됩니다.

In [ ]:
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), '런타임 유형이 GPU로 설정 안 됐습니다. 런타임 > 런타임 유형 변경에서 GPU(T4)를 선택하세요.'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/tag_lens_training'

In [ ]:
import zipfile, os

for name in ['dataset', 'dataset_daynight']:
    zip_path = f'{DRIVE_DIR}/{name}.zip'
    assert os.path.exists(zip_path), f'{zip_path} 가 없습니다. Drive에 업로드했는지 확인하세요.'
    print(f'{name}.zip 압축 해제 중...')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall('/content')
    print(f'  -> /content/{name} 완료')

## 1. 장르 5-class CNN 학습

`scripts/train_genre_model.py`와 동일: MobileNetV2 전이학습, 1단계(base 동결, 15 epoch) → 2단계(상위 30레이어 해동, 최대 30 epoch fine-tuning). 클래스 가중치는 실제 폴더 이미지 수에서 자동 계산.

In [ ]:
from pathlib import Path
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

GENRE_DATASET_DIR = Path('/content/dataset')
GENRE_MODEL_PATH = Path('/content/models/genre_model.h5')
GENRE_MODEL_PATH.parent.mkdir(exist_ok=True, parents=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_FROZEN = 15
EPOCHS_FINETUNE = 30
GENRE_CLASS_NAMES = ['자연', '도시', '음식', '사물', '인물']


def build_genre_data_pipeline():
    train_gen = keras.preprocessing.image.ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True,
        zoom_range=0.1,
        brightness_range=[0.8, 1.2],
    )
    val_gen = keras.preprocessing.image.ImageDataGenerator(rescale=1.0 / 255)

    train_ds = train_gen.flow_from_directory(
        str(GENRE_DATASET_DIR / 'train'), target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', classes=GENRE_CLASS_NAMES, shuffle=True,
    )
    val_ds = val_gen.flow_from_directory(
        str(GENRE_DATASET_DIR / 'val'), target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', classes=GENRE_CLASS_NAMES, shuffle=False,
    )
    return train_ds, val_ds


def compute_class_weights(train_ds):
    counts = np.bincount(train_ds.classes, minlength=len(GENRE_CLASS_NAMES))
    total = counts.sum()
    weights = {i: float(total / (len(GENRE_CLASS_NAMES) * c)) for i, c in enumerate(counts)}
    for name, idx in train_ds.class_indices.items():
        print(f'  {name}: {counts[idx]}장 -> weight {weights[idx]:.3f}')
    return weights


def build_genre_model():
    base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = False

    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(len(GENRE_CLASS_NAMES), activation='softmax')(x)
    return keras.Model(inputs, outputs), base

In [ ]:
train_ds, val_ds = build_genre_data_pipeline()
print(f'훈련 샘플: {train_ds.samples}  |  검증 샘플: {val_ds.samples}')
print(f'클래스 인덱스: {train_ds.class_indices}')
print('클래스 가중치:')
class_weights = compute_class_weights(train_ds)

model, base = build_genre_model()
model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
print('[1단계] base 동결, 분류 헤드 학습')
callbacks_phase1 = [
    EarlyStopping(patience=6, restore_best_weights=True, verbose=1),
    ModelCheckpoint(str(GENRE_MODEL_PATH), save_best_only=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]
model.fit(train_ds, epochs=EPOCHS_FROZEN, validation_data=val_ds, callbacks=callbacks_phase1, class_weight=class_weights)

In [ ]:
print('[2단계] fine-tuning (base 상위 30레이어 해동)')
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

callbacks_phase2 = [
    EarlyStopping(patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint(str(GENRE_MODEL_PATH), save_best_only=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]
model.fit(train_ds, epochs=EPOCHS_FINETUNE, validation_data=val_ds, callbacks=callbacks_phase2)

loss, acc = model.evaluate(val_ds, verbose=0)
print(f'최종 검증 정확도: {acc:.4f}  |  손실: {loss:.4f}')

## 2. 주간/야간 이진 CNN 학습

`scripts/train_daynight_model.py`와 동일: MobileNetV2 전이학습, 1단계(15 epoch) → 2단계(상위 30레이어 해동, 최대 20 epoch).

In [ ]:
DAYNIGHT_DATASET_DIR = Path('/content/dataset_daynight')
DAYNIGHT_MODEL_PATH = Path('/content/models/daynight_model.h5')

DN_BATCH_SIZE = 16
DN_EPOCHS_FROZEN = 15
DN_EPOCHS_FINETUNE = 20
DAYNIGHT_CLASS_NAMES = ['주간', '야간']


def build_daynight_data_pipeline():
    train_gen = keras.preprocessing.image.ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True,
        zoom_range=0.1,
        brightness_range=[0.9, 1.1],
    )
    val_gen = keras.preprocessing.image.ImageDataGenerator(rescale=1.0 / 255)

    train_ds = train_gen.flow_from_directory(
        str(DAYNIGHT_DATASET_DIR / 'train'), target_size=IMG_SIZE, batch_size=DN_BATCH_SIZE,
        class_mode='categorical', classes=DAYNIGHT_CLASS_NAMES, shuffle=True,
    )
    val_ds = val_gen.flow_from_directory(
        str(DAYNIGHT_DATASET_DIR / 'val'), target_size=IMG_SIZE, batch_size=DN_BATCH_SIZE,
        class_mode='categorical', classes=DAYNIGHT_CLASS_NAMES, shuffle=False,
    )
    return train_ds, val_ds


def build_daynight_model():
    base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = False

    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(len(DAYNIGHT_CLASS_NAMES), activation='softmax')(x)
    return keras.Model(inputs, outputs), base


dn_train_ds, dn_val_ds = build_daynight_data_pipeline()
print(f'훈련 샘플: {dn_train_ds.samples}  |  검증 샘플: {dn_val_ds.samples}')
print(f'클래스 인덱스: {dn_train_ds.class_indices}')

dn_model, dn_base = build_daynight_model()
dn_model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
dn_model.summary()

In [ ]:
print('[1단계] base 동결, 분류 헤드 학습')
dn_callbacks_phase1 = [
    EarlyStopping(patience=6, restore_best_weights=True, verbose=1),
    ModelCheckpoint(str(DAYNIGHT_MODEL_PATH), save_best_only=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]
dn_model.fit(dn_train_ds, epochs=DN_EPOCHS_FROZEN, validation_data=dn_val_ds, callbacks=dn_callbacks_phase1)

In [ ]:
print('[2단계] fine-tuning (base 상위 30레이어 해동)')
dn_base.trainable = True
for layer in dn_base.layers[:-30]:
    layer.trainable = False

dn_model.compile(optimizer=keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

dn_callbacks_phase2 = [
    EarlyStopping(patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint(str(DAYNIGHT_MODEL_PATH), save_best_only=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]
dn_model.fit(dn_train_ds, epochs=DN_EPOCHS_FINETUNE, validation_data=dn_val_ds, callbacks=dn_callbacks_phase2)

dn_loss, dn_acc = dn_model.evaluate(dn_val_ds, verbose=0)
print(f'최종 검증 정확도: {dn_acc:.4f}  |  손실: {dn_loss:.4f}')

## 3. 결과 모델을 Drive로 복사 (다운로드용)

In [ ]:
import shutil

shutil.copy(GENRE_MODEL_PATH, f'{DRIVE_DIR}/genre_model.h5')
shutil.copy(DAYNIGHT_MODEL_PATH, f'{DRIVE_DIR}/daynight_model.h5')
print(f'저장 완료: {DRIVE_DIR}/genre_model.h5, {DRIVE_DIR}/daynight_model.h5')
print('이제 Drive에서 두 파일을 다운로드해서 로컬 tag_lens/models/ 에 덮어쓰면 됩니다.')